# Parkinson's true-OFF score — one notebook, from baselines to the best RMSE

**Goal.** For every patient visit, predict the *true OFF* MDS-UPDRS motor score (`target`).
**Metric.** RMSE (lower is better). **Validation.** 5-fold cross-validation **grouped by patient**,
because the Kaggle test set contains *new patients* only.

### The story in one minute
1. A model that only sees one visit at a time is stuck at **RMSE ≈ 7.4**: the measured `on`/`off` scores are very noisy.
2. But the true OFF score of a patient moves **smoothly with age** (almost a straight line). So the other visits of the
   same patient are extra measurements of the same curve. Averaging them out removes most of the noise.
   All visits of a test patient are in `X_test`, so we can use them (we only use *features*, never the target).
3. We therefore build **patient-context features** and a **two-stage model** (stage 1 cleans each visit, stage 2 smooths along the patient's trajectory).

| Section | What happens |
|---|---|
| 0 | Setup |
| 1 | Load data |
| 2 | Quick EDA (what the data looks like and why patient context matters) |
| 3 | Validation protocol |
| 4 | The lab baselines (dummy, Ridge, HistGradientBoosting, skrub `tabular_pipeline`) |
| 5 | Feature engineering |
| 6 | The patient-context models (single-stage and two-stage) |
| 7 | Honest cross-validation of every model + diagnostics |
| 8 | Final fit and Kaggle CSV files (`submissions/<report-key>.csv`) |
| 9 | Push the reports to Skore Hub |
| 10 | Summary |

## 0. Setup

In [ ]:
import json
import os
import time
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline

try:  # CatBoost is optional: without it the final ensemble simply has two members instead of three
    from catboost import CatBoostRegressor
except ImportError:
    CatBoostRegressor = None

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25})

# --- project paths: find the repo root (the folder that contains data/X_train.csv) ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "X_train.csv").exists())
DATA_DIR = ROOT / "data"
SUBMISSIONS_DIR = ROOT / "submissions"
RESULTS_DIR = ROOT / "results"
SUBMISSIONS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# --- global settings ---
SEED = 42
N_SPLITS = 5                      # patient-grouped CV folds
PROJECT_NAME = "bobathon-esilv"   # Skore Hub project
PUSH_TO_HUB = True                # set to False to skip section 9
print("Repo root:", ROOT)

## 1. Load data

* `X_train` / `y_train`: one row per visit, aligned on `Index`.
* `X_test`: visits of **other patients** (no patient appears in both files).

In [ ]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Safety checks: rows are aligned and train/test patients are disjoint.
assert (X_train["Index"].values == y_train["Index"].values).all()
assert not set(X_train["patient_id"]) & set(X_test["patient_id"])
y = y_train["target"].reset_index(drop=True)

print(f"train: {len(X_train):,} visits, {X_train['patient_id'].nunique():,} patients")
print(f"test : {len(X_test):,} visits, {X_test['patient_id'].nunique():,} patients")
print(f"visits per patient (train): min {X_train.groupby('patient_id').size().min()}, "
      f"max {X_train.groupby('patient_id').size().max()}")
X_train.head()

## 2. Quick EDA

Three things matter for the modelling choices below.

In [ ]:
# (a) Missing values: `off` is missing for ~42% of visits, `on` for ~30%, and never both missing.
missing = pd.DataFrame({"train": X_train.isna().mean(), "test": X_test.isna().mean()}).drop(index="Index")
missing = missing[missing.max(axis=1) > 0].sort_values("train")
ax = (missing * 100).plot.barh(figsize=(7, 3.5), color=["#4c72b0", "#dd8452"])
ax.set_xlabel("% of visits with a missing value")
ax.set_title("Missing values (kept as signal, never dropped)")
plt.tight_layout(); plt.show()

In [ ]:
# (b) The true OFF score of one patient is a SMOOTH curve in age, while the measured scores are very noisy.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
example_patients = X_train.groupby("patient_id").size().loc[lambda s: s >= 10].index[[3, 50, 120]]
for ax, pid in zip(axes, example_patients):
    rows = X_train["patient_id"] == pid
    ax.plot(X_train.loc[rows, "age"], y[rows], "-o", color="black", ms=3, label="true OFF (target)")
    ax.scatter(X_train.loc[rows, "age"], X_train.loc[rows, "off"], marker="x", color="#c44e52", label="measured off")
    ax.scatter(X_train.loc[rows, "age"], X_train.loc[rows, "on"], marker="s", s=14, color="#4c72b0", label="measured on")
    ax.set_title(f"patient {pid}"); ax.set_xlabel("age")
axes[0].set_ylabel("MDS-UPDRS motor score"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# (c) How noisy is a single visit, and how smooth is the target along one patient?
visits = X_train.assign(target=y)
off_ok = visits["off"].notna() & (visits["target"] > 20)
ratio = (visits["off"] / visits["target"])[off_ok]
print(f"measured off / true OFF : mean {ratio.mean():.2f}, std {ratio.std():.2f}  -> one visit is a very noisy measurement")


def quad_residual_sd(g):
    """Residual standard deviation of a parabola fitted to the target of ONE patient."""
    p = np.polyfit(g["age"] - g["age"].mean(), g["target"], 2)
    return np.sqrt(np.sum((g["target"] - np.polyval(p, g["age"] - g["age"].mean())) ** 2) / (len(g) - 3))


smooth = visits.groupby("patient_id").filter(lambda g: len(g) >= 6).groupby("patient_id").apply(quad_residual_sd)
print(f"parabola fitted on ONE patient's target -> residual sd {smooth.median():.2f} points "
      f"(the target std over all visits is {y.std():.1f})")
print(f"ON and OFF log-errors of the same visit are strongly correlated -> a visit carries ~one noisy measurement, not two.")

**Take-aways**
* A single visit is a noisy measurement (±25% of the true score), so a per-visit model plateaus around RMSE 7.4.
* The true score of a patient follows a very smooth curve of age → combining the patient's visits removes most of the noise.
* Raters (clinicians) add small systematic biases, so `rater_id` is kept as a categorical feature.
* Missing values are informative (ON-only visits are common), so we use tree models that handle NaN natively.

## 3. Validation protocol

`GroupKFold` puts **all visits of a patient in the same fold**, which mimics the Kaggle split.
The same folds are used for every model, so the scores are directly comparable.

In [ ]:
groups = X_train["patient_id"].reset_index(drop=True)
cv_splits = list(GroupKFold(n_splits=N_SPLITS).split(X_train, y, groups))
for k, (tr, va) in enumerate(cv_splits):
    assert not set(groups.iloc[tr]) & set(groups.iloc[va])        # no patient on both sides
print("Fold sizes (validation visits):", [len(va) for _, va in cv_splits])

results = {}        # report key -> dict of scores
oof_predictions = {}  # report key -> out-of-fold predictions (used for plots)


def score_oof(key, oof, description, seconds=np.nan):
    """Store RMSE/MAE per fold + mean for one model and print a one-line summary."""
    fold_rmse = [root_mean_squared_error(y.iloc[va], oof[va]) for _, va in cv_splits]
    fold_mae = [mean_absolute_error(y.iloc[va], oof[va]) for _, va in cv_splits]
    results[key] = dict(model=description, rmse=np.mean(fold_rmse), rmse_std=np.std(fold_rmse),
                        mae=np.mean(fold_mae), fold_rmse=fold_rmse, seconds=seconds)
    oof_predictions[key] = oof
    print(f"{key:<34} RMSE {np.mean(fold_rmse):6.3f} ± {np.std(fold_rmse):.3f}   MAE {np.mean(fold_mae):6.3f}   ({seconds:.0f}s)")


def cross_validate(key, description, make_model, X):
    """Fit a fresh model on 4 folds, predict the 5th, repeat. Returns out-of-fold predictions."""
    t0 = time.perf_counter()
    oof = np.zeros(len(y))
    for tr, va in cv_splits:
        model = make_model().fit(X.iloc[tr], y.iloc[tr])
        oof[va] = model.predict(X.iloc[va])
    score_oof(key, oof, description, time.perf_counter() - t0)
    return oof

## 4. The lab baselines

The guided lab (DAY1/DAY2) builds these in order. They all look at **one visit at a time**:
* `01_dummy`: always predicts the mean target (the floor every model must beat).
* `02_ridge`: a linear model on the numeric columns (missing values median-imputed).
* `03_hist_gradient_boosting`: trees that keep missing values as signal.
* `04_hist_gradient_boosting_tuned`: same, with better settings and a few extra columns.
* `05_tabular_pipeline`: skrub encodes the text columns (`cohort`, `gene`, `rater_id`) automatically.

In [ ]:
NUMERIC_COLS = ["sexM", "age_at_diagnosis", "age", "ledd", "time_since_intake_on",
                "time_since_intake_off", "on", "off"]
CATEGORICAL_COLS = ["cohort", "gene", "rater_id"]


def lab_features(df):
    """Numeric columns + three simple extra columns + categoricals as pandas 'category'."""
    out = df[NUMERIC_COLS].copy()
    out["disease_duration"] = df["age"] - df["age_at_diagnosis"]
    out["on_off_diff"] = df["off"] - df["on"]
    out["on_off_mean"] = (df["on"] + df["off"]) / 2
    for col in CATEGORICAL_COLS:
        out[col] = pd.Categorical(df[col])
    return out


lab_categories = {c: pd.Categorical(pd.concat([X_train[c], X_test[c]])).categories for c in CATEGORICAL_COLS}


def lab_features_aligned(df):
    out = lab_features(df)
    for col in CATEGORICAL_COLS:       # same category list for train and test
        out[col] = pd.Categorical(df[col], categories=lab_categories[col])
    return out


# Each model uses one feature set; a feature set is a function applied identically to train and test.
feature_sets = {
    "numeric": lambda df: df[NUMERIC_COLS],
    "lab": lab_features_aligned,
    "all_columns": lambda df: df.drop(columns=["Index", "patient_id"]),     # for skrub's tabular_pipeline
}
X_sets = {name: fn(X_train) for name, fn in feature_sets.items()}

TUNED_HGB = dict(max_iter=500, learning_rate=0.05, max_depth=5, min_samples_leaf=20,
                 l2_regularization=0.0, random_state=SEED)

lab_models = {
    "01_dummy": ("Dummy mean", lambda: DummyRegressor(strategy="mean"), "numeric"),
    "02_ridge": ("Ridge + median imputation",
                 lambda: make_pipeline(SimpleImputer(strategy="median"), Ridge(alpha=1.0)), "numeric"),
    "03_hist_gradient_boosting": ("HistGradientBoosting (default)",
                                  lambda: HistGradientBoostingRegressor(random_state=SEED), "numeric"),
    "04_hist_gradient_boosting_tuned": ("HistGradientBoosting (tuned) + extra columns",
                                        lambda: HistGradientBoostingRegressor(**TUNED_HGB), "lab"),
}
try:
    from skrub import tabular_pipeline
    lab_models["05_tabular_pipeline"] = ("skrub tabular_pipeline", lambda: tabular_pipeline("regressor"), "all_columns")
except ImportError:
    print("skrub is not installed: skipping 05_tabular_pipeline")

for key, (description, make_model, feature_set) in lab_models.items():
    cross_validate(key, description, make_model, X_sets[feature_set])

## 5. Feature engineering: give every visit the context of its patient

Every function below uses **features only** (never the target), so it is safe to apply to the test set,
where all visits of each patient are available.

**5.1 Visit-level columns.** The raw columns plus three derived ones (`dd` = years since diagnosis, ON/OFF difference and ratio).

**5.2 Patient-level columns.** For each patient, sorted by age, and for each of `on`, `off`, `ledd`:
mean / std / count, a straight line and a parabola fitted on the patient's visits (evaluated at every visit's age), the slope,
a linear interpolation, and the deviation of the visit from the fitted line.

**5.3 "Debiased" measurements `z`.** The ON score is lower than the OFF score because of the drug, and the effect depends
on `time_since_intake_on`. We estimate that effect curve from visits that have both ON and OFF, then divide `on` by it
so that ON and OFF are on the same scale. `z` is the average of the available ones.

**5.4 Stage-1 columns (used by the two-stage model).** Once stage 1 has produced a cleaned prediction `p1` for every visit,
we smooth it along each patient's trajectory and also expose the neighbouring visits (3 before / 3 after).

In [ ]:
BASE_COLS = ["cohort", "sexM", "gene", "age_at_diagnosis", "age", "ledd", "time_since_intake_on",
             "time_since_intake_off", "rater_id", "on", "off", "dd", "on_off_diff", "on_off_ratio"]
TSI_BINS = np.array([0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0])   # bins of time-since-intake (hours)


def patient_rows(df):
    """List of row-position arrays, one per patient, each sorted by age."""
    pid = df["patient_id"].to_numpy()
    order = np.lexsort((df["age"].to_numpy(float), pid))
    pid_sorted = pid[order]
    cuts = np.r_[0, np.flatnonzero(pid_sorted[1:] != pid_sorted[:-1]) + 1, len(order)]
    return [order[a:b] for a, b in zip(cuts[:-1], cuts[1:])]


def fit_curve(age, values, degree):
    """Fit a polynomial of `degree` to the non-missing `values` of ONE patient and evaluate it at every age.

    Returns (fitted values, slope). With too few points the patient mean is used instead."""
    ok = ~np.isnan(values)
    if ok.sum() == 0:
        return np.full(len(age), np.nan), np.nan
    if ok.sum() < degree + 2:
        return np.full(len(age), values[ok].mean()), np.nan
    x = age - age[ok].mean()
    coef = np.polyfit(x[ok], values[ok], degree)
    return np.polyval(coef, x), (coef[-2] if degree >= 1 else np.nan)


def fit_ratio_curve(df):
    """ON/OFF ratio as a function of time since the last drug intake (no target used)."""
    ok = df["on"].notna() & (df["off"] > 5) & df["time_since_intake_on"].notna()
    ratio = (df["on"] / df["off"])[ok].to_numpy()
    bin_id = np.digitize(df["time_since_intake_on"][ok].to_numpy(), TSI_BINS, right=True)
    curve = np.array([np.median(ratio[bin_id == b]) if (bin_id == b).sum() >= 20 else np.nan
                      for b in range(len(TSI_BINS) + 1)])
    plateau = np.median(ratio[df["time_since_intake_on"][ok].to_numpy() > 2.0])
    return np.where(np.isnan(curve), plateau, curve), plateau


def visit_features(df, categories):
    """5.1 Raw visit columns + dd / on_off_diff / on_off_ratio. Categoricals use a fixed category list."""
    out = df[[c for c in BASE_COLS if c in df.columns and c not in ("dd", "on_off_diff", "on_off_ratio")]].copy()
    for col in CATEGORICAL_COLS:
        out[col] = pd.Categorical(df[col], categories=categories[col])
    out["dd"] = df["age"] - df["age_at_diagnosis"]
    out["on_off_diff"] = df["off"] - df["on"]
    out["on_off_ratio"] = df["on"] / (df["off"] + 1.0)
    return out[BASE_COLS]


def patient_features(df, ratio_curve, plateau, off_scale):
    """5.2 + 5.3: summaries of each patient's other visits, and the debiased measurement z."""
    n = len(df)
    age, on, off, ledd = (df[c].to_numpy(float) for c in ("age", "on", "off", "ledd"))
    tsi_on = df["time_since_intake_on"].to_numpy(float)
    on_ratio = np.where(np.isnan(tsi_on), plateau, ratio_curve[np.digitize(np.nan_to_num(tsi_on), TSI_BINS, right=True)])
    z_on, z_off = on / on_ratio, off / off_scale
    z = np.where(np.isnan(z_on), z_off, np.where(np.isnan(z_off), z_on, (z_on + z_off) / 2))

    names = ["visit_num", "n_visits", "t_from_first", "t_to_last", "span", "z_on", "z_off", "z", "z_lin", "z_quad", "z_mean"]
    for c in ("on", "off", "ledd"):
        names += [f"{c}_{s}" for s in ("pmean", "pstd", "pcnt", "lin", "slope", "quad", "interp", "dev")]
    out = {name: np.full(n, np.nan) for name in names}
    out["z_on"], out["z_off"], out["z"] = z_on, z_off, z

    for rows in patient_rows(df):
        a, m = age[rows], len(rows)
        out["visit_num"][rows], out["n_visits"][rows] = np.arange(m), m
        out["t_from_first"][rows], out["t_to_last"][rows], out["span"][rows] = a - a[0], a[-1] - a, a[-1] - a[0]
        for c, v in (("on", on[rows]), ("off", off[rows]), ("ledd", ledd[rows])):
            ok = ~np.isnan(v)
            if ok.any():
                out[f"{c}_pmean"][rows], out[f"{c}_pcnt"][rows] = v[ok].mean(), ok.sum()
                out[f"{c}_interp"][rows] = np.interp(a, a[ok], v[ok])
            else:
                out[f"{c}_pcnt"][rows] = 0
            if ok.sum() > 1:
                out[f"{c}_pstd"][rows] = v[ok].std()
            lin, slope = fit_curve(a, v, 1)
            out[f"{c}_lin"][rows], out[f"{c}_slope"][rows] = lin, slope
            out[f"{c}_quad"][rows] = fit_curve(a, v, 2)[0]
            out[f"{c}_dev"][rows] = v - lin
        zr = z[rows]
        out["z_lin"][rows], out["z_quad"][rows] = fit_curve(a, zr, 1)[0], fit_curve(a, zr, 2)[0]
        if (~np.isnan(zr)).any():
            out["z_mean"][rows] = np.nanmean(zr)
    return pd.DataFrame(out, index=df.index)


def stage1_features(df, p1, n_neighbours=3):
    """5.4 Smooth the stage-1 predictions `p1` along each patient's trajectory + neighbouring visits."""
    n, age = len(df), df["age"].to_numpy(float)
    ks = [k for k in range(-n_neighbours, n_neighbours + 1) if k != 0]
    out = {name: np.full(n, np.nan) for name in ["p1_lin", "p1_quad", "p1_mean", "p1_std", "p1_slope"]}
    out.update({f"p1_lag{k}": np.full(n, np.nan) for k in ks})
    out.update({f"p1_dt{k}": np.full(n, np.nan) for k in ks})
    for rows in patient_rows(df):
        a, v, m = age[rows], p1[rows], len(rows)
        out["p1_lin"][rows], out["p1_slope"][rows] = fit_curve(a, v, 1)
        out["p1_quad"][rows] = fit_curve(a, v, 2)[0]
        out["p1_mean"][rows], out["p1_std"][rows] = v.mean(), v.std()
        for k in ks:                       # value and time gap of the k-th neighbouring visit (k<0: earlier)
            src = np.arange(m) + k
            ok = (src >= 0) & (src < m)
            out[f"p1_lag{k}"][rows[ok]] = v[src[ok]]
            out[f"p1_dt{k}"][rows[ok]] = a[src[ok]] - a[ok]
    result = pd.DataFrame(out, index=df.index)
    result.insert(0, "p1", p1)
    return result


demo = patient_features(X_train.head(500), *fit_ratio_curve(X_train), off_scale=0.83)
print(f"{demo.shape[1]} patient-level columns, e.g.:", list(demo.columns[:8]), "...")

## 6. The patient-context models

`PatientRegressor` is a normal scikit-learn estimator: `fit(X, y)` / `predict(X)` on the raw table
(it needs the `patient_id` column to know which visits belong together).

* **Single-stage** (`two_stage=False`): one LightGBM on visit columns + patient-level columns.
* **Two-stage** (`two_stage=True`):
  1. *Stage 1* is a LightGBM that only sees one visit at a time. Its **out-of-fold** predictions `p1` (computed with an inner
     patient-grouped CV, so no row is predicted by a model that saw its own target) are a cleaned-up score for each visit.
  2. *Stage 2* sees everything, plus `p1` smoothed along the patient's trajectory and the neighbouring visits' `p1`.
* **Ensemble**: stage 2 can average several learners: two LightGBMs with *linear trees* (they fit straight lines inside each leaf,
  which suits a smooth, almost linear trajectory) and a CatBoost. Different learners make different errors, so the average is better.

In [ ]:
LEARNERS = {
    # name: (class, parameters). Hyper-parameters were tuned with grouped CV and are fixed here (no early stopping on validation data).
    "lgbm_plain": (LGBMRegressor, dict(n_estimators=600, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                                        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0)),
    "lgbm_linear_a": (LGBMRegressor, dict(n_estimators=500, learning_rate=0.03, num_leaves=31, min_child_samples=20,
                                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                                           linear_tree=True, linear_lambda=1.0)),
    "lgbm_linear_b": (LGBMRegressor, dict(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                                           linear_tree=True, linear_lambda=1.0)),
}
if CatBoostRegressor is not None:
    LEARNERS["catboost"] = (CatBoostRegressor, dict(iterations=1500, learning_rate=0.06, depth=8, l2_leaf_reg=3, verbose=0))
STAGE1_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                     subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0)


def make_learner(name, seed):
    cls, params = LEARNERS[name]
    seed_arg = "random_seed" if name == "catboost" else "random_state"
    extra = {} if name == "catboost" else {"verbose": -1}
    return cls(**params, **{seed_arg: seed}, **extra)


def for_catboost(F):
    """CatBoost wants categoricals as strings (with an explicit 'NA' level)."""
    F = F.copy()
    for col in CATEGORICAL_COLS:
        F[col] = F[col].astype(object).where(F[col].notna(), "NA").astype(str)
    return F


class PatientRegressor(BaseEstimator, RegressorMixin):
    """Patient-context regressor (see the text above). `X` must contain `patient_id`; `Index` is ignored."""

    def __init__(self, two_stage=True, learners=("lgbm_linear_a", "lgbm_linear_b", "catboost"), random_state=SEED):
        self.two_stage = two_stage
        self.learners = learners
        self.random_state = random_state

    # ---- label-free features -------------------------------------------------------------------------------------
    def _features(self, X):
        return pd.concat([visit_features(X, self.categories_),
                          patient_features(X, self.ratio_curve_, self.plateau_, self.off_scale_)], axis=1)

    def fit(self, X, y):
        X, y = X.reset_index(drop=True), np.asarray(y, dtype=float)
        self.learners_used_ = [n for n in self.learners if n in LEARNERS]          # skips catboost if it is not installed
        self.categories_ = {c: pd.Categorical(X[c]).categories for c in CATEGORICAL_COLS}
        self.ratio_curve_, self.plateau_ = fit_ratio_curve(X)
        both = X["off"].notna() & (X["off"] > 0) & (y > 10)
        self.off_scale_ = float(np.median(X.loc[both, "off"] / y[both]))            # OFF scores are ~83% of the true OFF
        F = self._features(X)

        if self.two_stage:
            p1 = np.zeros(len(X))                                                    # out-of-fold stage-1 predictions
            for tr, va in GroupKFold(n_splits=5).split(F, y, X["patient_id"]):
                m = LGBMRegressor(**STAGE1_PARAMS, random_state=self.random_state, verbose=-1).fit(F.iloc[tr][BASE_COLS], y[tr])
                p1[va] = m.predict(F.iloc[va][BASE_COLS])
            self.stage1_ = LGBMRegressor(**STAGE1_PARAMS, random_state=self.random_state, verbose=-1).fit(F[BASE_COLS], y)
            F = pd.concat([F, stage1_features(X, p1)], axis=1)

        self.columns_ = list(F.columns)
        self.models_ = {}
        for name in self.learners_used_:
            model = make_learner(name, self.random_state)
            if name == "catboost":
                model.fit(for_catboost(F), y, cat_features=CATEGORICAL_COLS)
            else:
                model.fit(F, y)
            self.models_[name] = model
        return self

    def predict_components(self, X):
        """Predictions of every learner, as a dict {learner name: array}."""
        X = X.reset_index(drop=True)
        F = self._features(X)
        if self.two_stage:
            F = pd.concat([F, stage1_features(X, self.stage1_.predict(F[BASE_COLS]))], axis=1)
        F = F[self.columns_]
        return {name: m.predict(for_catboost(F) if name == "catboost" else F) for name, m in self.models_.items()}

    def predict(self, X):
        return np.mean(list(self.predict_components(X).values()), axis=0)       # simple average of the learners


MODELS = {   # report key -> (description, estimator factory)
    "07_patient_features_lgbm": ("Single stage: LightGBM + patient-level features",
                                 lambda: PatientRegressor(two_stage=False, learners=("lgbm_plain",))),
    "08_two_stage_lgbm": ("Two stages: LightGBM linear trees",
                          lambda: PatientRegressor(two_stage=True, learners=("lgbm_linear_a",))),
    "09_two_stage_ensemble": ("Two stages: ensemble of linear-tree LightGBMs + CatBoost",
                              lambda: PatientRegressor(two_stage=True)),
}

## 7. Honest cross-validation of the patient-context models

For each outer fold the whole pipeline (features, **stage 1 with its inner CV**, stage 2) is fitted on the 4 training folds only
and evaluated on the 5th, i.e. on patients it has never seen. Nothing from the validation patients leaks into any stage.

The two-stage models share the same stage 1, so we fit the ensemble once per fold and read the single-learner model `08`
from the ensemble's components (identical to fitting it alone).

In [ ]:
# --- 07: single stage ---
cross_validate("07_patient_features_lgbm", MODELS["07_patient_features_lgbm"][0],
               MODELS["07_patient_features_lgbm"][1], X_train)

# --- 08 and 09: two stages (one ensemble fit per fold gives both) ---
t0 = time.perf_counter()
oof_components = {}
for fold, (tr, va) in enumerate(cv_splits):
    model = MODELS["09_two_stage_ensemble"][1]().fit(X_train.iloc[tr], y.iloc[tr])
    for name, pred in model.predict_components(X_train.iloc[va]).items():
        oof_components.setdefault(name, np.zeros(len(y)))[va] = pred
    print(f"  fold {fold + 1}/{N_SPLITS} done ({time.perf_counter() - t0:.0f}s)")
elapsed = time.perf_counter() - t0

score_oof("08_two_stage_lgbm", oof_components["lgbm_linear_a"], MODELS["08_two_stage_lgbm"][0], elapsed)
score_oof("09_two_stage_ensemble", np.mean(list(oof_components.values()), axis=0), MODELS["09_two_stage_ensemble"][0], elapsed)
for name, oof in oof_components.items():
    print(f"   ensemble member {name:<14} RMSE {np.mean([root_mean_squared_error(y.iloc[va], oof[va]) for _, va in cv_splits]):.3f}")

In [ ]:
# Summary of every model, best first.
summary = (pd.DataFrame(results).T
           .assign(rmse=lambda d: d.rmse.astype(float), rmse_std=lambda d: d.rmse_std.astype(float), mae=lambda d: d.mae.astype(float))
           [["model", "rmse", "rmse_std", "mae"]].sort_values("rmse"))
summary["gain_vs_dummy_%"] = (100 * (1 - summary["rmse"] / summary.loc["01_dummy", "rmse"])).round(1)
display(summary.round(3))

fig, ax = plt.subplots(figsize=(8, 3.8))
order = summary.sort_values("rmse", ascending=False)
ax.barh(order.index, order["rmse"], xerr=order["rmse_std"], color="#4c72b0")
for i, v in enumerate(order["rmse"]):
    ax.text(v + 0.25, i, f"{v:.2f}", va="center", fontsize=9)
ax.set_xlabel("RMSE (patient-grouped 5-fold CV, lower is better)")
plt.tight_layout(); plt.show()

In [ ]:
# Diagnostics for the best model: predicted vs true, and where the error comes from.
best_key = summary.index[0]
best_oof = oof_predictions[best_key]
errors = y.values - best_oof

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].scatter(y, best_oof, s=2, alpha=0.15, color="#4c72b0")
axes[0].plot([0, 110], [0, 110], "k--", lw=1)
axes[0].set(xlabel="true OFF score", ylabel="predicted (out-of-fold)", title=f"{best_key}")

n_off = X_train.groupby("patient_id")["off"].transform("count")      # how many OFF measurements the patient has
by_n = pd.Series(errors ** 2).groupby(n_off.clip(upper=10).values).mean() ** 0.5
axes[1].bar(by_n.index.astype(str), by_n.values, color="#55a868")
axes[1].set(xlabel="number of OFF measurements of the patient (10 = 10 or more)", ylabel="RMSE", title="More measurements → smaller error")

by_level = pd.Series(errors ** 2).groupby(pd.cut(y, [-1, 15, 25, 35, 45, 60, 120]).values).mean() ** 0.5
axes[2].bar([str(i) for i in by_level.index], by_level.values, color="#c44e52")
axes[2].set(xlabel="true OFF score", ylabel="RMSE", title="Error grows with severity")
axes[2].tick_params(axis="x", labelsize=7)
plt.tight_layout(); plt.show()

## 8. Final fit and Kaggle files

Every model is fitted on **all** training visits and predicts the test visits.
Each file is named after its Skore Hub report key (`submissions/<key>.csv`) with the columns `Index,target`.

In [ ]:
def write_submission(key, predictions):
    """Write submissions/<key>.csv and check it has exactly the format Kaggle expects."""
    sub = pd.DataFrame({"Index": X_test["Index"].values, "target": np.asarray(predictions, dtype=float)})
    assert list(sub.columns) == list(sample_submission.columns)
    assert len(sub) == len(sample_submission) and (sub["Index"].values == sample_submission["Index"].values).all()
    assert sub["target"].notna().all() and np.isfinite(sub["target"]).all()
    sub.to_csv(SUBMISSIONS_DIR / f"{key}.csv", index=False)
    return sub


fitted = {}
t0 = time.perf_counter()
for key, (description, make_model, feature_set) in lab_models.items():
    fitted[key] = make_model().fit(X_sets[feature_set], y)
    write_submission(key, fitted[key].predict(feature_sets[feature_set](X_test)))

test_components = {}
for key in ("07_patient_features_lgbm", "09_two_stage_ensemble"):
    fitted[key] = MODELS[key][1]().fit(X_train, y)
    test_components[key] = fitted[key].predict_components(X_test)
    write_submission(key, np.mean(list(test_components[key].values()), axis=0))
write_submission("08_two_stage_lgbm", test_components["09_two_stage_ensemble"]["lgbm_linear_a"])
print(f"Final fits done in {time.perf_counter() - t0:.0f}s")
print("Files written:", sorted(p.name for p in SUBMISSIONS_DIR.glob("*.csv")))

In [ ]:
# Sanity checks on the best file + which features matter most in the best model.
best_file = pd.read_csv(SUBMISSIONS_DIR / f"{best_key}.csv")
print(best_file.describe().round(2).T[["count", "mean", "std", "min", "max"]])
print(f"train target mean {y.mean():.2f} / std {y.std():.2f}  vs  test prediction mean {best_file.target.mean():.2f}")

stage2 = fitted["09_two_stage_ensemble"]
gain = pd.Series(stage2.models_["lgbm_linear_a"].booster_.feature_importance("gain"), index=stage2.columns_)
ax = (gain / gain.sum()).sort_values().tail(15).plot.barh(figsize=(7, 4.5), color="#4c72b0")
ax.set_xlabel("share of total gain (stage-2 LightGBM)"); plt.tight_layout(); plt.show()

## 9. Push the reports to Skore Hub

Credentials and workspace are read from the `.skore` file at the repo root (the key is never printed).
Each model gets one `EstimatorReport` evaluated on a **patient-grouped hold-out** (the first CV fold), and is stored with
`project.put(<report key>, report)`. Keys that already exist in the project are skipped so nothing is overwritten
(Kaggle needs a *new* key for every submission). If the Hub cannot be reached, the cell explains why and carries on.

In [ ]:
def hub_is_reachable(url, timeout=8):
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except urllib.error.HTTPError:
        return True          # the server answered (e.g. 401/404): it is reachable
    except Exception as exc:
        print(f"Skore Hub not reachable from here ({type(exc).__name__}).")
        return False


def push_reports_to_hub():
    import skore

    config = json.loads((ROOT / ".skore").read_text())
    os.environ["SKORE_HUB_API_KEY"] = config["api_key"]                  # skore reads the key from the environment
    os.environ.setdefault("SKORE_HUB_URI", config["hub_url"])
    if not hub_is_reachable(config["hub_url"]):
        return {}
    if hasattr(skore, "login"):                                           # older skore versions need an explicit login
        skore.login(mode="hub")
    project = skore.Project(name=PROJECT_NAME, mode="hub", workspace=config["workspace"])
    print(f"Connected to workspace '{config['workspace']}', project '{PROJECT_NAME}'")

    try:
        existing = set(project.summarize().reset_index().get("key", []))
    except Exception:
        existing = set()

    tr, va = cv_splits[0]                                                 # patient-grouped hold-out
    jobs = {key: (make_model, X_sets[feature_set]) for key, (_, make_model, feature_set) in lab_models.items()}
    jobs.update({key: (make_model, X_train) for key, (_, make_model) in MODELS.items()})
    urls = {}
    for key, (make_model, X_used) in jobs.items():
        if key in existing:
            print(f"  {key}: already on the Hub, skipped")
            continue
        t0 = time.perf_counter()
        try:
            report = skore.EstimatorReport(make_model(), X_train=X_used.iloc[tr], y_train=y.iloc[tr],
                                           X_test=X_used.iloc[va], y_test=y.iloc[va])
            project.put(key, report)
            urls[key] = f"https://skore.probabl.ai/{config['workspace']}/projects/{PROJECT_NAME}/reports/{key}"
            print(f"  {key}: pushed in {time.perf_counter() - t0:.0f}s")
        except Exception as exc:
            print(f"  {key}: FAILED ({type(exc).__name__}: {str(exc)[:150]})")
    return urls


hub_urls = {}
if PUSH_TO_HUB and (ROOT / ".skore").exists():
    try:
        hub_urls = push_reports_to_hub()
    except Exception as exc:
        print(f"Hub push skipped: {type(exc).__name__}: {str(exc)[:200]}")
else:
    print("Hub push disabled or .skore not found.")

## 10. Summary

In [ ]:
summary["csv_file"] = [f"submissions/{k}.csv" for k in summary.index]
summary["hub_report"] = [hub_urls.get(k, "not pushed") for k in summary.index]
summary.round(4).to_csv(RESULTS_DIR / "cv_summary.csv", index_label="report_key")
display(summary.round(3))
print(f"Best model: {best_key}  ->  RMSE {summary.loc[best_key, 'rmse']:.3f} "
      f"(dummy {summary.loc['01_dummy', 'rmse']:.3f}, best lab model {summary.loc[[k for k in summary.index if k[:2] in ('02','03','04','05')], 'rmse'].min():.3f})")
print("Kaggle file to upload:", f"submissions/{best_key}.csv")